In [ ]:
pip install medmnist albumentations timm

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import torch
import random
import itertools
import time
import seaborn as sns



# Dataset
from medmnist import BloodMNIST
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

# Train test split
from sklearn.model_selection import train_test_split

# Model & Hyperparameters
import torch.nn as nn
import torch.nn.functional as F
import timm
import torch.optim as optim

# Evaluation Metrics
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [ ]:
torch.cuda.is_available()

In [ ]:
CLASS_NAMES = [
    'Basophil', 'Eosinophil', 'Erythroblast', 'Immature Granulocyte',
    'Lymphocyte', 'Monocyte', 'Neutrophil', 'Platelet'
]
NUM_CLASSES = 8

## Load BloodMNIST at 128x128 resolution

In [ ]:
train_data = BloodMNIST(split = 'train', download = True, size = 128)
val_data = BloodMNIST(split = 'val', download = True, size = 128)
test_data = BloodMNIST(split = 'test', download = True, size = 128)

In [ ]:
print("Train dataset size: ", len(train_data))
print("Validation dataset size: ", len(val_data))
print("Test dataset size: ", len(test_data))

# Data Exploration

## Single Image Array & Corresponding Label

In [ ]:
sample_img, sample_label = train_data[0]
print("=== Single Sample ===")
print(f"Type : {type(sample_img)}")
print(f"Mode : {sample_img.mode}")
print(f"Size : {sample_img.size}")

In [ ]:
sample_arr = np.array(sample_img)
# print(sample_arr)
print(f"As numpy array: {sample_arr.shape}")
print(f"Dtype: {sample_arr.dtype}")
print(f"Pixel range: [{sample_arr.min()}, {sample_arr.max()}]")
print(f"Label : {sample_label} --> {CLASS_NAMES[int(sample_label)]}")

## Per Channel Statistics (Mean, Min, Max)

In [ ]:
# --- Per-channel stats on a single image ---
print("\n=== Per-Channel Stats (single image) ===")
for i, ch in enumerate(['R', 'G', 'B']):
    ch_data = sample_arr[:, :, i]
    print(f"  {ch} — mean: {ch_data.mean():.2f}, std: {ch_data.std():.2f}, "
          f"min: {ch_data.min()}, max: {ch_data.max()}")


In [ ]:
all_indices = np.arange(len(train_data))
all_labels  = np.array([int(label) for _, label in train_data])

In [ ]:
all_indices

In [ ]:
all_labels

## Label Distribution in the BloodMNIST Dataset

In [ ]:
# --- Dataset-level label distribution ---
print("\n=== Label Distribution (train set) ===")
unique, counts = np.unique(all_labels, return_counts=True)
total = len(all_labels)
for cls, cnt in zip(unique, counts):
    bar = '█' * (cnt // 50)
    print(f"  {CLASS_NAMES[cls]:<28} : {cnt:>5} ({cnt/total*100:.1f}%)  {bar}")


In [ ]:
# --- Class imbalance ratio ---
print(f"\n  Most frequent class  : {CLASS_NAMES[counts.argmax()]} ({counts.max()})")
print(f"  Least frequent class : {CLASS_NAMES[counts.argmin()]} ({counts.min()})")
print(f"  Imbalance ratio      : {counts.max() / counts.min():.2f}x")

In [ ]:
counts

In [ ]:
# --- Pixel intensity distribution per channel across full train set ---
# Sample 500 images for speed
sample_size = 500
sample_idx  = np.random.choice(len(train_data), sample_size, replace=False)
sample_imgs = np.stack([np.array(train_data[i][0]) for i in sample_idx], axis=0) / 255.0

print("\n=== Per-Channel Stats (500-image sample) ===")
for i, ch in enumerate(['R', 'G', 'B']):
    ch_data = sample_imgs[:, :, :, i]
    print(f"  {ch} — mean: {ch_data.mean():.4f}, std: {ch_data.std():.4f}, "
          f"min: {ch_data.min():.4f}, max: {ch_data.max():.4f}")

In [ ]:
# --- Visualize pixel intensity histograms per channel ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['red', 'green', 'blue']
for i, (ch, color) in enumerate(zip(['R', 'G', 'B'], colors)):
    axes[i].hist(sample_imgs[:, :, :, i].ravel(), bins=50, color=color, alpha=0.7)
    axes[i].set_title(f'Channel {ch} — pixel intensity distribution')
    axes[i].set_xlabel('Pixel value (0–1)')
    axes[i].set_ylabel('Frequency')
plt.suptitle('BloodMNIST — pixel intensity histograms (500-image sample)', fontsize=12)
plt.tight_layout()
plt.show()

# --- Matrix view: print one channel as a small grid ---
print("\n=== Matrix View: R channel of sample image (top-left 8×8 patch) ===")
print(sample_arr[:8, :8, 0])  # raw uint8 values

In [ ]:
print(sample_img)
print(sample_label)

## Normalize images for smooth gradient flow

In [ ]:
train_images = np.stack([np.array(img) for img, _ in train_data], axis = 0)
print(type(train_images))
print(train_images.shape)

## Compute per channel mean and std

In [ ]:
train_images = train_images/255.0

In [ ]:
# train_images

## Visualize Sample Images Per Class

In [ ]:
SAMPLES_PER_CLASS = 3

In [ ]:
class_indices = {c:[] for c in range(NUM_CLASSES)}
class_indices

In [ ]:
for idx, (_, label) in enumerate(train_data):
    label = int(label)
    if len(class_indices[label]) < SAMPLES_PER_CLASS:
        class_indices[label].append(idx)
    if all(len(v) == SAMPLES_PER_CLASS for v in class_indices.values()):
        break
    
fig, axes = plt.subplots(NUM_CLASSES, SAMPLES_PER_CLASS, figsize=(9, 24))
for cls in range(NUM_CLASSES):
    for s, idx in enumerate(class_indices[cls]):
        img, _ = train_data[idx]
        axes[cls][s].imshow(np.array(img))
        axes[cls][s].axis('off')
        if s == 0:
            axes[cls][s].set_title(CLASS_NAMES[cls], fontsize=9, loc='left')
plt.suptitle('BloodMNIST — 3 samples per class', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()
    

# Train Test Split to Simulate Semi-supervised Learning

In [ ]:
all_indices = np.arange(len(train_data))
all_labels = np.array([int(label) for _, label in train_data])

In [ ]:
labeled_indices, unlabeled_indices = train_test_split(
    all_indices,
    test_size=0.99,
    stratify=all_labels,
    random_state=42
)

In [ ]:
labeled_indices

In [ ]:
print(f"Labeled   : {len(labeled_indices)}")
print(f"Unlabeled : {len(unlabeled_indices)}")

# Per-class count in labeled set
print("\nPer-class count in labeled set:")
labeled_labels = all_labels[labeled_indices]
for cls in range(NUM_CLASSES):
    count = (labeled_labels == cls).sum()
    print(f"  {CLASS_NAMES[cls]:<28} : {count}")

# Setting Up Dataset and Dataloaders

In [ ]:
class LabeledDataset(Dataset):
    """
    Returns (weakly_augmented_image, label) for labeled training samples.
    """
    def __init__(self, indices, data, transform):
        self.indices   = indices
        self.data      = data
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        img, label = self.data[self.indices[i]]
        return self.transform(img), int(label.item())


In [ ]:
class UnlabeledDataset(Dataset):
    """
    Returns (weak_aug_image, strong_aug_image) for unlabeled training samples.
    No label is returned — used for pseudo-label consistency loss.
    """
    def __init__(self, indices, data, weak_transform, strong_transform):
        self.indices          = indices
        self.data             = data
        self.weak_transform   = weak_transform
        self.strong_transform = strong_transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        img, _ = self.data[self.indices[i]]
        return self.weak_transform(img), self.strong_transform(img)

In [ ]:
class TestDataset(Dataset):
    """
    Returns (image, label) for val/test evaluation. No augmentation beyond normalize.
    """
    def __init__(self, data, transform):
        self.data      = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        img, label = self.data[i]
        return self.transform(img), int(label.item())

## Define Augmentations

In [ ]:
MEAN = train_images.mean(axis = (0, 1, 2))
STD = train_images.std(axis = (0, 1, 2))
MEAN_UINT8 = (int(MEAN[0]*255), int(MEAN[1]*255), int(MEAN[2]*255))

In [ ]:
print(f"MEAN calculated: {MEAN}")
print(f"STD calculated:  {STD}")
print(f"MEAN_UINT8 calculated for padding fill: {MEAN_UINT8}")

In [ ]:
# Weak aug — used now; strong aug defined in Phase 2
weak_transform = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.Pad(padding =16, fill=MEAN_UINT8),
    T.RandomCrop(128, padding=16),
    T.ToTensor(),
    T.Normalize(MEAN, STD)
])

eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(MEAN, STD)
])


In [ ]:
# Random strong transform
rand_strong_transform = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.Pad(padding=16, fill=MEAN_UINT8),
    T.RandomCrop(128),
    T.RandAugment(num_ops=2, magnitude=9),
    T.ToTensor(),
    T.Normalize(MEAN.tolist(), STD.tolist()),
    T.RandomErasing(p=1.0, scale=(0.0625, 0.0625),
    ratio=(1.0, 1.0), value=0)
    ])

In [ ]:
class AlbumentationsAdapter:
    """
    Bridges the gap between PyTorch PIL inputs and Albumentations NumPy inputs [14].
    Ensures that your Albumentations pipeline behaves exactly like a torchvision 
    transform, returning a normalized PyTorch Tensor [14].
    """
    def __init__(self, albumentations_transform):
        self.transform = albumentations_transform

    def __call__(self, img):
        # 1. Convert PIL Image to numpy array (HWC, uint8)
        img_np = np.array(img)
        
        # 2. Apply Albumentations transform (handles normalization & ToTensorV2)
        augmented = self.transform(image=img_np)
        
        # 3. Return the normalized Tensor
        return augmented['image']

In [ ]:
hema_strong_pipeline = A.Compose([
    # Spatial/Morphological Layer
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    # Use BORDER_REFLECT to mirror background, avoiding black border artifacts [4]
    A.Rotate(limit=180, border_mode=cv2.BORDER_REFLECT, p=0.5),
    A.ElasticTransform(alpha=80, sigma=8, p=0.5),

    # Optical/Staining Layer
    # Hue limit set strictly to 0.02 to prevent destroying clinical color meanings [4]
    A.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, 
                  hue=0.1, p=0.8),
    A.GaussianBlur(blur_limit=3, p=0.2),
    
    # Normalization & Tensor Conversion
    A.Normalize(mean=MEAN.tolist(), std=STD.tolist()),

    
    # Occlusion Layer (Cutout) - Updated for Albumentations 2.x
    A.CoarseDropout(
        num_holes_range=(1, 4),      # Replaces min_holes and max_holes
        hole_height_range=(8, 20),   # Replaces min_height and max_height
        hole_width_range=(8, 20),    # Replaces min_width and max_width
        fill=0,                      # Replaces fill_value
        p=0.5
    ),
    ToTensorV2()
])

# Wrap the Albumentations pipeline in our Adapter class for clean dataset execution
hema_strong_transform = AlbumentationsAdapter(hema_strong_pipeline)

In [ ]:
# -------------------------------------------------------
# DataLoaders
# -------------------------------------------------------

labeled_dataset   = LabeledDataset(labeled_indices, train_data, weak_transform)
unlabeled_dataset = UnlabeledDataset(unlabeled_indices, train_data,
                                     weak_transform, rand_strong_transform)
val_dataset       = TestDataset(val_data,  eval_transform)
test_dataset      = TestDataset(test_data, eval_transform)

labeled_loader   = DataLoader(labeled_dataset,   batch_size=16, shuffle=True,  drop_last=True,  num_workers=2)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=64, shuffle=True,  drop_last=True,  num_workers=2)
val_loader       = DataLoader(val_dataset,       batch_size=64, shuffle=False, drop_last=False, num_workers=2)
test_loader      = DataLoader(test_dataset,      batch_size=64, shuffle=False, drop_last=False, num_workers=2)

# Cycling unlabeled loader for training loop
unlabeled_cycle = itertools.cycle(unlabeled_loader)

# --- Sanity check ---
imgs_l, labels_l = next(iter(labeled_loader))
imgs_wu, imgs_su = next(iter(unlabeled_loader))

print(f"\nLabeled batch   — images: {imgs_l.shape}, labels: {labels_l.shape}")
print(f"Unlabeled batch — weak: {imgs_wu.shape}, strong: {imgs_su.shape}")
print(f"Val batch       — images: {next(iter(val_loader))[0].shape}")

In [ ]:
# ============================================================
# SECTION 1 — End-to-End Data Contract Verification
# ============================================================

def verify_batch(name, batch, expected_image_shape, expect_label=True):
    if expect_label:
        imgs, labels = batch
        print(f"\n{name}")
        print(f"  images : {imgs.shape}, dtype={imgs.dtype}")
        print(f"  labels : {labels.shape}, dtype={labels.dtype}")
        assert imgs.shape[1:] == expected_image_shape, f"Wrong image shape: {imgs.shape}"
        assert labels.dtype == torch.long, f"Labels must be long, got {labels.dtype}"
        assert labels.ndim == 1, f"Labels must be 1D, got shape {labels.shape}"
    else:
        weak, strong = batch
        print(f"\n{name}")
        print(f"  weak   : {weak.shape}, dtype={weak.dtype}")
        print(f"  strong : {strong.shape}, dtype={strong.dtype}")
        assert weak.shape == strong.shape, "Weak and strong shapes must match"
        assert weak.shape[1:] == expected_image_shape, f"Wrong image shape: {weak.shape}"

EXPECTED = torch.Size([3, 128, 128])

verify_batch("LabeledLoader",   next(iter(labeled_loader)),   EXPECTED, expect_label=True)
verify_batch("UnlabeledLoader", next(iter(unlabeled_loader)), EXPECTED, expect_label=False)
verify_batch("ValLoader",       next(iter(val_loader)),       EXPECTED, expect_label=True)
verify_batch("TestLoader",      next(iter(test_loader)),      EXPECTED, expect_label=True)

# Check pixel value range after normalization — should be roughly [-3, 3]
imgs, _ = next(iter(labeled_loader))
print(f"\nNormalized pixel range : [{imgs.min():.3f}, {imgs.max():.3f}]")
print(f"Expected               : roughly [-3.5, 3.5]")

# Check label range
_, labels = next(iter(labeled_loader))
print(f"\nLabel range : [{labels.min()}, {labels.max()}]")
print(f"Expected    : [0, 7]")

print("\n✓ All checks passed — data pipeline is clean")

In [ ]:

# ============================================================
# ResNet-18 (Primary Backbone) Setup
# ============================================================

def get_resnet18_backbone(num_classes=8):
    """
    Instantiates a non-pretrained ResNet-18 model via timm.
    The final fully connected linear layer is automatically configured 
    to output logits for your target classes (8 for BloodMNIST).
    """
    model = timm.create_model(
        'resnet18', 
        pretrained=True, 
        num_classes=num_classes
    )
    return model

def count_parameters(model):
    """
    Calculates the total number of active, trainable parameters.
    """
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [ ]:
# 1. Instantiate the model
model = get_resnet18_backbone(num_classes=8)

# 2. Count parameters to verify network capacity
total_params = count_parameters(model)
print(f"Model Parameter Count: {total_params / 1e6:.2f}M parameters (Expected: ~11.2M)")

# 3. Move model to your Kaggle P100 GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Model successfully moved to: {device}")


In [ ]:
# ============================================================
# Penultimate Feature Extraction Verification (GradCAM)
# ============================================================

# Let's perform a validation dry run to verify the model's forward paths.
# We pass a dummy batch of 2 normalized images of size 128x128.
model.eval()
with torch.no_grad():
    dummy_input = torch.randn(2, 3, 128, 128).to(device)
    
    # Test Path 1: Standard classification forward pass
    dummy_output = model(dummy_input)
    
    # Test Path 2: Feature map extraction right before global pooling.
    # timm natively supports this via 'forward_features'.
    dummy_features = model.forward_features(dummy_input)


In [ ]:
# Display verification results
print("\n=== Model Data Contract Verification ===")
print(f"Input batch shape:           {dummy_input.shape}")
print(f"Classification logits shape: {dummy_output.shape} (Expected: [batch_size, 8])")
print(f"Penultimate feature shape:   {dummy_features.shape} (Expected: [batch_size, 512, 4, 4])")
print("========================================")
print("✔ Data contract verified successfully!")


## Shared Utilities

In [ ]:
# ============================================================
# Shared Evaluation Utility
# ============================================================

def evaluate_model(model, dataloader, device):
    """
    Evaluates the model on a given dataset.
    Returns: accuracy, macro_f1, true_labels, predicted_labels [4.3].
    """
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for imgs, labels in dataloader:
            imgs = imgs.to(device)
            # Forward pass
            logits = model(imgs)
            preds = torch.argmax(logits, dim=1)
            
            # Store predictions and true labels
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            
    # Calculate target metrics
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    
    return acc, macro_f1, all_labels, all_preds

In [ ]:
def plot_colorful_confusion_matrix(true_labels, pred_labels, class_names, run_name, save_path=None):
    """
    Plots a high-resolution, colorful confusion matrix using Seaborn.
    Saves the image as a 300 DPI print-ready PNG if save_path is provided [10.3].
    """
    # 1. Compute the raw confusion matrix array
    cm = confusion_matrix(true_labels, pred_labels)
    
    # 2. Set up the figure with high resolution
    plt.figure(figsize=(10, 8), dpi=150)
    
    # 3. Create the heatmap
    # We use "Purples" to match the Giemsa stain aesthetic of your blood cells!
    sns.heatmap(
        cm, 
        annot=True, 
        fmt="d", 
        cmap="Purples", 
        xticklabels=class_names, 
        yticklabels=class_names,
        cbar=True,
        square=True,
        annot_kws={"size": 11, "weight": "bold"}  # Bold text inside cells for readability
    )
    
    # 4. Add titles and labels with appropriate padding and font sizes
    plt.title(f"Confusion Matrix: {run_name}", fontsize=14, pad=20, weight="bold")
    plt.xlabel("Predicted Cell Type", fontsize=12, labelpad=10, weight="bold")
    plt.ylabel("True Cell Type", fontsize=12, labelpad=10, weight="bold")
    
    # 5. Rotate class names on axes so they never overlap
    plt.xticks(rotation=45, ha="right", fontsize=10)
    plt.yticks(rotation=0, fontsize=10)
    
    # 6. Adjust layout and save/show
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)  # 300 DPI is standard for print publications
        print(f"  ✔ Publication-ready figure saved to: {save_path}")
    plt.show()

In [ ]:
import os
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score
from IPython.display import display

def style_max(s):
    """
    Custom styling function to highlight the maximum value in each column.
    Uses an extremely soft, matte slate-gray background, deep charcoal text,
    and bold typography for a professional, academic look.
    """
    is_max = s == s.max()
    # Return CSS style rules for the maximum cells, empty string for others
    return [
        'background-color: #f1f5f9; color: #0f172a; font-weight: bold;' if v else '' 
        for v in is_max
    ]

def log_run_results(run_name, true_labels, pred_labels, test_acc, test_f1, csv_path="hema_results_summary.csv"):
    """
    Programmatically calculates overall and class-specific metrics, saves them 
    to a persistent CSV file, and displays a styled table in the notebook.
    """
    # 1. Compute macro precision and recall
    macro_prec = precision_score(true_labels, pred_labels, average='macro', zero_division=0)
    macro_rec = recall_score(true_labels, pred_labels, average='macro', zero_division=0)
    
    # 2. Compute per-class F1-scores
    class_f1s = f1_score(true_labels, pred_labels, average=None, zero_division=0)
    
    # 3. Create a dictionary for this run's results
    new_row = {
        "Method": run_name,
        "Accuracy": test_acc,
        "Macro Precision": macro_prec,
        "Macro Recall": macro_rec,
        "Macro F1": test_f1
    }
    
    # Add per-class F1 columns dynamically
    for i, class_name in enumerate(CLASS_NAMES):
        new_row[f"{class_name} F1"] = class_f1s[i]
        
    # 4. Load existing CSV or initialize a new one
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
    else:
        df = pd.DataFrame(columns=list(new_row.keys()))
        
    # 5. Prevent duplicates
    if run_name in df["Method"].values:
        idx = df[df["Method"] == run_name].index[0]
        for col, val in new_row.items():
            df.at[idx, col] = val
        print(f"✔ Updated existing metrics for '{run_name}' in {csv_path}.")
    else:
        new_row_df = pd.DataFrame([new_row])
        df = pd.concat([df, new_row_df], ignore_index=True)
        print(f"✔ Successfully logged new metrics for '{run_name}' to {csv_path}.")
        
    # 6. Save back to persistent CSV file
    df.to_csv(csv_path, index=False)
    
    # 7. Set up visual styling for the Jupyter notebook
    format_dict = {
        "Accuracy": "{:.2%}",
        "Macro Precision": "{:.4f}",
        "Macro Recall": "{:.4f}",
        "Macro F1": "{:.4f}"
    }
    for class_name in CLASS_NAMES:
        format_dict[f"{class_name} F1"] = "{:.4f}"
        
    # Apply custom, ultra-soft styling to columns (excluding the 'Method' column)
    styled_df = df.style.format(format_dict).apply(
        style_max, 
        subset=[col for col in df.columns if col != "Method"], 
        axis=0
    )
    
    print("\n=== PERSISTENT METHODOLOGY LEADERBOARD ===")
    display(styled_df)
    print("==========================================\n")
    
    return df


In [ ]:
class FreeMatchThreshold:
    """
    Self-adaptive per-class thresholding for FixMatch.
    Replaces global tau=0.95 with per-class thresholds
    that adapt based on each class's learning progress.
    """
    def __init__(self, num_classes=8, momentum=0.9, device='cuda'):
        self.num_classes = num_classes
        self.momentum    = momentum
        self.device      = device
        # Initialize uniformly — 1/num_classes each
        self.class_thresh = torch.ones(num_classes, device=device) / num_classes
        self.history      = []  # for logging

    def update(self, probs):
        """probs: softmax output (batch_size, num_classes)"""
        with torch.no_grad():
            batch_mean = probs.mean(dim=0)
            self.class_thresh = (self.momentum * self.class_thresh
                                 + (1 - self.momentum) * batch_mean)

    def get_mask(self, probs, pseudo_labels):
        """Returns float mask where per-class confidence threshold is met."""
        with torch.no_grad():
            global_thresh  = self.class_thresh.mean()
            normalized     = self.class_thresh / (global_thresh + 1e-6)
            confidence, _  = probs.max(dim=1)
            sample_thresh  = normalized[pseudo_labels]
            mask = confidence.ge(global_thresh * sample_thresh).float()
        return mask

    def log_thresholds(self, epoch, class_names):
        global_thresh = self.class_thresh.mean().item()
        print(f"\n[FreeMatch] Epoch {epoch} — Per-class thresholds "
              f"(global mean: {global_thresh:.4f}):")
        snapshot = {}
        for cls, thresh in enumerate(self.class_thresh):
            t = thresh.item()
            flag = " ←" if t < global_thresh else ""
            print(f"  {class_names[cls]:<28}: {t:.4f}{flag}")
            snapshot[class_names[cls]] = t
        self.history.append({'epoch': epoch, **snapshot})

In [ ]:
def test_freematch():
    """
    Unit tests for FreeMatchThreshold.
    Run once before training to verify correctness.
    """
    print("=== FreeMatch Sanity Tests ===\n")
    fm = FreeMatchThreshold(num_classes=8, momentum=0.9, device='cpu')

    # Test 1: Initial thresholds are uniform
    expected = 1.0 / 8
    assert torch.allclose(fm.class_thresh,
                          torch.full((8,), expected), atol=1e-6), \
        "FAIL: Initial thresholds not uniform"
    print("✓ Test 1 passed: Initial thresholds uniform (0.125 each)")

    # Test 2: Update moves threshold toward batch mean
    # Simulate model always confident about class 0
    fake_probs = torch.zeros(16, 8)
    fake_probs[:, 0] = 0.9
    fake_probs[:, 1:] = 0.1 / 7
    fm.update(fake_probs)
    assert fm.class_thresh[0] > fm.class_thresh[1], \
        "FAIL: Class 0 threshold should be highest after update"
    print("✓ Test 2 passed: Threshold updates toward batch distribution")

    # Test 3: Rare class gets lower threshold (more pseudo-labels allowed)
    fm2 = FreeMatchThreshold(num_classes=8, momentum=0.9, device='cpu')
    # Simulate 50 steps: model rarely confident about class 5 (Monocyte)
    for _ in range(50):
        probs = torch.ones(16, 8) * 0.05
        probs[:, 0] = 0.65   # model loves class 0
        probs[:, 5] = 0.01   # model avoids class 5
        probs = probs / probs.sum(dim=1, keepdim=True)
        fm2.update(probs)
    assert fm2.class_thresh[5] < fm2.class_thresh[0], \
        "FAIL: Rare class should have lower threshold"
    print("✓ Test 3 passed: Rare class gets lower threshold (less starvation)")

    # Test 4: Mask is float and correct shape
    probs        = torch.softmax(torch.randn(32, 8), dim=1)
    pseudo       = probs.argmax(dim=1)
    fm2.update(probs)
    mask = fm2.get_mask(probs, pseudo)
    assert mask.shape == (32,),       "FAIL: Wrong mask shape"
    assert mask.dtype == torch.float32, "FAIL: Mask must be float32"
    assert mask.max() <= 1.0 and mask.min() >= 0.0, "FAIL: Mask out of [0,1]"
    print("✓ Test 4 passed: Mask shape, dtype, and range correct")

    # Test 5: Mask ratio is reasonable (not 0%, not 100%)
    probs_high = torch.softmax(torch.randn(64, 8) * 3, dim=1)  # sharper
    pseudo_high = probs_high.argmax(dim=1)
    mask_high = fm2.get_mask(probs_high, pseudo_high)
    ratio = mask_high.mean().item()
    assert 0.0 < ratio < 1.0, f"FAIL: Degenerate mask ratio {ratio:.2f}"
    print(f"✓ Test 5 passed: Mask ratio is non-degenerate ({ratio*100:.1f}%)")

    print("\n=== All tests passed — FreeMatch is safe to use ===\n")

In [ ]:
def plot_threshold_evolution(free_thresh, class_names, save_path='freematch_thresholds.png'):
    if not free_thresh.history:
        print("No threshold history to plot.")
        return
    
    epochs = [h['epoch'] for h in free_thresh.history]
    fig, ax = plt.subplots(figsize=(12, 5))
    
    for cls in class_names:
        vals = [h[cls] for h in free_thresh.history]
        # Highlight problem classes
        lw    = 2.5 if cls in ['Monocyte', 'Immature Granulocyte'] else 1.0
        style = '-'  if cls in ['Monocyte', 'Immature Granulocyte'] else '--'
        ax.plot(epochs, vals, label=cls, linewidth=lw, linestyle=style)
    
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Per-class threshold')
    ax.set_title('FreeMatch — Per-class Threshold Evolution')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Saved to {save_path}")

## Fully Supervised Baseline with 1% Training Data from the Labeled Set

In [ ]:
run_name = "Supervised Baseline"

In [ ]:

# ============================================================
#  Training Setup & Parameters
# ============================================================

# Target hyperparameters exactly matching the reference plan [4.1]
EPOCHS = 300
CHECKPOINT_NAME = "supervised_resnet18_best.pth"

# Freshly re-initialize model for Phase 4 to ensure clean baseline weights [3.1]
model = get_resnet18_backbone(num_classes=8).to(device)

criterion = nn.CrossEntropyLoss()

# Standard paper optimizer: SGD + Momentum + Weight Decay [2.4, 4.1]
optimizer = optim.SGD(
    model.parameters(), 
    lr=0.03, 
    momentum=0.9, 
    weight_decay=1e-4
)

# Cosine Annealing Scheduler over the full 200 epochs [2.4, 4.1]
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
# ============================================================
# SECTION 4.2 — Training Loop
# ============================================================

print("--- Starting Supervised Baseline Training (10% Labeled Data Only) ---")
best_val_acc = 0.0
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    
    # Train exclusively on 10% labeled loader [1.5]
    for imgs, labels in labeled_loader:
        # Move inputs to your T4 GPU
        imgs = imgs.to(device)
        labels = labels.squeeze().long().to(device)  # Squeeze handles MedMNIST label shapes [4.2]
        
        # Standard gradient step
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * imgs.size(0)
        
    # Decay learning rate [4.2]
    scheduler.step()
    
    epoch_loss = running_loss / len(labeled_loader.dataset)
    
    # Evaluate and print metrics every 10 epochs [4.2]
    if epoch % 10 == 0 or epoch == 1:
        val_acc, val_f1, _, _ = evaluate_model(model, val_loader, device)
        print(f"Epoch {epoch:03d}/{EPOCHS:03d} | Train Loss: {epoch_loss:.4f} | "
              f"Val Acc: {val_acc*100:.2f}% | Val F1: {val_f1:.4f} | LR: {scheduler.get_last_lr()[0]:.5f}")
        
        # Save best model checkpoint based on validation accuracy [4.1, 4.2]
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'val_f1': val_f1
            }, CHECKPOINT_NAME)
            print(f"  ✔ New best model saved to {CHECKPOINT_NAME} (Val Acc: {val_acc*100:.2f}%)")

training_duration = time.time() - start_time
print(f"\nTraining completed in {training_duration/60:.2f} minutes!")
print(f"Best Validation Accuracy achieved: {best_val_acc*100:.2f}%")


In [ ]:
# ============================================================
# Baseline Evaluation on Test Set
# ============================================================

print("\n--- Running Final Evaluation on Test Set ---")

# 1. Load the best saved checkpoint [4.3]
checkpoint = torch.load(CHECKPOINT_NAME)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best checkpoint from epoch {checkpoint['epoch']} with Val Acc {checkpoint['val_acc']*100:.2f}%")

# 2. Run evaluation on test set [4.3]
test_acc, test_f1, true_labels, pred_labels = evaluate_model(model, test_loader, device)

# 3. Compile and report metrics [4.3]
print("\n=== Baseline Supervised Test Set Performance ===")
print(f"Overall Test Accuracy : {test_acc*100:.2f}%")
print(f"Macro F1-Score        : {test_f1:.4f}")
print("================================================")

print("\nDetailed Per-Class Performance:")
print(classification_report(true_labels, pred_labels, target_names=CLASS_NAMES))

# Define an automatic, descriptive filename for saving
save_filename = f"{run_name.lower().replace(' ', '_').replace('+', 'and')}_confusion_matrix.png"

# Generate the beautiful colorful plot
print("\nGenerating Confusion Matrix Plot...")
plot_colorful_confusion_matrix(
    true_labels=true_labels, 
    pred_labels=pred_labels, 
    class_names=CLASS_NAMES, 
    run_name=run_name,
    save_path=save_filename
)

log_run_results(
    run_name=run_name,
    true_labels=true_labels,
    pred_labels=pred_labels,
    test_acc=test_acc,
    test_f1=test_f1
)

## Standard FixMatch Baseline

In [ ]:
EPOCHS = 300

In [ ]:
run_name = "Standard FixMatch"

In [ ]:
test_freematch()

In [ ]:
# ============================================================
# FixMatch Training Utilities
# ============================================================

def get_lambda_ramp(epoch, ramp_epochs=20):
    """
    Linearly ramps up the unlabeled loss weight lambda_u 
    from 0.0 to 1.0 over the first N epochs [5.1].
    """
    if epoch < ramp_epochs:
        return float(epoch) / float(ramp_epochs)
    return 1.0

# Define the best checkpoint name for Standard FixMatch
FIXMATCH_CHECKPOINT = "fixmatch_resnet18_best.pth"

# Freshly re-initialize model for Phase 5 to ensure clean weights [3.1]
model = get_resnet18_backbone(num_classes=8).to(device)

criterion_supervised = nn.CrossEntropyLoss()

# Standard paper optimizer: SGD + Momentum + Weight Decay [2.4, B.1]
optimizer = optim.SGD(
    model.parameters(), 
    lr=0.03, 
    momentum=0.9, 
    weight_decay=1e-4
)

# Cosine Annealing Scheduler over the full 200 epochs [2.4, B.1]
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)

In [ ]:
# ============================================================
#  FixMatch Training Loop
# ============================================================

print("--- Starting Standard FixMatch Training (Baseline SSL) ---")
best_val_acc = 0.0
start_time = time.time()

# Create the cycle generator for the larger unlabeled loader [5.1]
unlabeled_cycle = itertools.cycle(unlabeled_loader)

free_thresh = FreeMatchThreshold(
num_classes=8, momentum=0.9, device=device)

for epoch in range(1, EPOCHS + 1):
    model.train()
    
    running_loss_s = 0.0
    running_loss_u = 0.0
    running_mask_ratio = 0.0
    steps_in_epoch = 0
    
    # Standard FixMatch step: zip labeled loader with the cycled unlabeled loader [5.1]
    for (labeled_imgs, labels), (weak_unlabeled, strong_unlabeled) in zip(labeled_loader, unlabeled_cycle):
        
        # Move inputs to target device (T4 GPU) [5.1]
        labeled_imgs = labeled_imgs.to(device)
        labels = labels.squeeze().long().to(device)
        
        weak_unlabeled = weak_unlabeled.to(device)
        strong_unlabeled = strong_unlabeled.to(device)
        
        # --------------------------------------------------------
        # Step 1: Calculate Supervised Loss (Weak Augmented Labeled)
        # --------------------------------------------------------
        logits_labeled = model(labeled_imgs)
        loss_supervised = criterion_supervised(logits_labeled, labels)
        
        # --------------------------------------------------------
        # Step 2: Generate Pseudo-Labels (Weak Augmented Unlabeled)
        # Must run under torch.no_grad() to block gradient tracking [5.1]
        # --------------------------------------------------------
        with torch.no_grad():
            logits_weak    = model(weak_unlabeled)
            probs_weak     = torch.softmax(logits_weak, dim=1)
            confidence, pseudo_labels = probs_weak.max(dim=1)
    
            # FreeMatch — update per-class thresholds
            free_thresh.update(probs_weak)
    
            # FreeMatch — get adaptive mask instead of fixed 0.95
            mask = free_thresh.get_mask(probs_weak, pseudo_labels)
            
        # --------------------------------------------------------
        # Step 3: Calculate Consistency Loss (Strong Augmented Unlabeled)
        # --------------------------------------------------------
        logits_strong = model(strong_unlabeled)
        
        # Standard Cross-Entropy calculated per sample, multiplied by the active mask [5.1]
        raw_loss_u = F.cross_entropy(logits_strong, pseudo_labels, reduction='none')
        loss_unsupervised = (raw_loss_u * mask).mean()
        
        # --------------------------------------------------------
        # Step 4: Combine Losses with Lambda Ramp-Up [5.1]
        # --------------------------------------------------------
        lambda_u = get_lambda_ramp(epoch, ramp_epochs=20)
        total_loss = loss_supervised + lambda_u * loss_unsupervised
        
        # Gradient Update Step
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        
        # Track running metrics
        running_loss_s += loss_supervised.item()
        running_loss_u += loss_unsupervised.item()
        running_mask_ratio += mask.mean().item()  # Percentage of batch above 0.95
        steps_in_epoch += 1
        
    # Decay learning rate [4.2]
    scheduler.step()
    
    # Calculate epoch-level averages
    epoch_loss_s = running_loss_s / steps_in_epoch
    epoch_loss_u = running_loss_u / steps_in_epoch
    epoch_mask_ratio = running_mask_ratio / steps_in_epoch
    
    # Evaluate and print metrics every 10 epochs [5.1]
    if epoch % 10 == 0 or epoch == 1:
        val_acc, val_f1, _, _ = evaluate_model(model, val_loader, device)
        print(f"Epoch {epoch:03d}/{EPOCHS} | Loss S: {epoch_loss_s:.4f} | Loss U: {epoch_loss_u:.4f} | "
              f"Mask Ratio: {epoch_mask_ratio*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Val F1: {val_f1:.4f}")
        if epoch % 20 == 0:
            free_thresh.log_thresholds(epoch, CLASS_NAMES)
        
        # Save best model checkpoint based on validation accuracy [5.1]
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'val_f1': val_f1,
                'mask_ratio': epoch_mask_ratio
            }, FIXMATCH_CHECKPOINT)
            print(f"  ✔ New best model saved to {FIXMATCH_CHECKPOINT} (Val Acc: {val_acc*100:.2f}%)")

training_duration = time.time() - start_time
print(f"\nStandard FixMatch training completed in {training_duration/60:.2f} minutes!")
print(f"Best Validation Accuracy achieved: {best_val_acc*100:.2f}%")

In [ ]:
plot_threshold_evolution(free_thresh, CLASS_NAMES)

In [ ]:
EPOCHS = 300

In [ ]:
# ============================================================
# Final Baseline SSL Evaluation
# ============================================================

print("\n--- Running Final Evaluation on Test Set ---")

# 1. Load the best saved checkpoint [4.3]
checkpoint = torch.load(FIXMATCH_CHECKPOINT)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best checkpoint from epoch {checkpoint['epoch']} with Val Acc {checkpoint['val_acc']*100:.2f}%")

# 2. Run evaluation on test set [4.3]
test_acc, test_f1, true_labels, pred_labels = evaluate_model(model, test_loader, device)

# 3. Compile and report metrics [4.3]
print("\n=== Standard FixMatch Test Set Performance ===")
print(f"Overall Test Accuracy : {test_acc*100:.2f}%")
print(f"Macro F1-Score        : {test_f1:.4f}")
print("================================================")

print("\nDetailed Per-Class Performance:")
print(classification_report(true_labels, pred_labels, target_names=CLASS_NAMES))

# Define an automatic, descriptive filename for saving
save_filename = f"{run_name.lower().replace(' ', '_').replace('+', 'and')}_confusion_matrix.png"

# Generate the beautiful colorful plot
print("\nGenerating Confusion Matrix Plot...")
plot_colorful_confusion_matrix(
    true_labels=true_labels, 
    pred_labels=pred_labels, 
    class_names=CLASS_NAMES, 
    run_name=run_name,
    save_path=save_filename
)

log_run_results(
    run_name=run_name,
    true_labels=true_labels,
    pred_labels=pred_labels,
    test_acc=test_acc,
    test_f1=test_f1
)

## HemaMatch + DA

## Hema-Aug

In [ ]:
run_name = "Hema-Aug Only"

In [ ]:
# ============================================================
# SECTION 6.0 — Rebuild Unlabeled Loader with Hema-Aug
# ============================================================

# We swap the strong transform from standard RandAugment to Hema-Aug [6.1]
hema_unlabeled_dataset = UnlabeledDataset(
    unlabeled_indices, 
    train_data,
    weak_transform,             # Kept identical to Phase 5
    hema_strong_transform       # Our custom AlbumentationsAdapter pipeline [4]
)

# Re-instantiate the dataloader with the new Hema-Aug dataset
hema_unlabeled_loader = DataLoader(
    hema_unlabeled_dataset, 
    batch_size=64,              # Kept identical [1.5]
    shuffle=True, 
    drop_last=True, 
    num_workers=2
)

# Re-create the cycle generator using the new Hema-Aug loader
hema_unlabeled_cycle = itertools.cycle(hema_unlabeled_loader)


In [ ]:
# ============================================================
# SECTION 6.1 — Phase 6 Training Setup
# ============================================================

HEMA_AUG_ONLY_CHECKPOINT = "hemamatch_aug_only_best.pth"

# Freshly re-initialize model for Phase 6 to ensure clean weights [3.1]
model = get_resnet18_backbone(num_classes=8).to(device)

criterion_supervised = nn.CrossEntropyLoss()

# Standard paper optimizer (identical to Phase 5) [6.1]
optimizer = optim.SGD(
    model.parameters(), 
    lr=0.03, 
    momentum=0.9, 
    weight_decay=1e-4
)

# Cosine Annealing Scheduler (identical to Phase 5) [6.1]
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)


In [ ]:
# ============================================================
# SECTION 6.2 — Hema-Aug Training Loop
# ============================================================

print("--- Starting HemaMatch Training: Phase 6 (Hema-Aug with Adaptive Thresholding) ---")
best_val_acc = 0.0
start_time = time.time()
free_thresh = FreeMatchThreshold(
num_classes=8, momentum=0.9, device=device)

for epoch in range(1, 301):
    model.train()
    
    running_loss_s = 0.0
    running_loss_u = 0.0
    running_mask_ratio = 0.0
    steps_in_epoch = 0
    
    # Zip labeled loader with the new Hema-Aug cycle [6.1]
    for (labeled_imgs, labels), (weak_unlabeled, strong_unlabeled) in zip(labeled_loader, hema_unlabeled_cycle):
        
        labeled_imgs = labeled_imgs.to(device)
        labels = labels.squeeze().long().to(device)
        
        weak_unlabeled = weak_unlabeled.to(device)
        strong_unlabeled = strong_unlabeled.to(device)
        
        # Step 1: Supervised Loss
        logits_labeled = model(labeled_imgs)
        loss_supervised = criterion_supervised(logits_labeled, labels)
        
        # Step 2: Pseudo-Label Generation (Weak Unlabeled)
        with torch.no_grad():
            logits_weak    = model(weak_unlabeled)
            probs_weak     = torch.softmax(logits_weak, dim=1)
            confidence, pseudo_labels = probs_weak.max(dim=1)
    
            # FreeMatch — update per-class thresholds
            free_thresh.update(probs_weak)
    
            # FreeMatch — get adaptive mask instead of fixed 0.95
            mask = free_thresh.get_mask(probs_weak, pseudo_labels)
    
            
            
        # Step 3: Consistency Loss using Hema-Aug (Strong Unlabeled)
        logits_strong = model(strong_unlabeled)
        raw_loss_u = F.cross_entropy(logits_strong, pseudo_labels, reduction='none')
        loss_unsupervised = (raw_loss_u * mask).mean()
        
        # Step 4: Combined Loss with Lambda Ramp-Up
        lambda_u = get_lambda_ramp(epoch, ramp_epochs=20)
        total_loss = loss_supervised + lambda_u * loss_unsupervised
        
        # Gradient Update
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        
        running_loss_s += loss_supervised.item()
        running_loss_u += loss_unsupervised.item()
        running_mask_ratio += mask.mean().item()
        steps_in_epoch += 1
        
    scheduler.step()
    
    epoch_loss_s = running_loss_s / steps_in_epoch
    epoch_loss_u = running_loss_u / steps_in_epoch
    epoch_mask_ratio = running_mask_ratio / steps_in_epoch
    
    # Evaluate every 10 epochs [6.2]
    if epoch % 10 == 0 or epoch == 1:
        val_acc, val_f1, _, _ = evaluate_model(model, val_loader, device)
        print(f"Epoch {epoch:03d}/300 | Loss S: {epoch_loss_s:.4f} | Loss U: {epoch_loss_u:.4f} | "
              f"Mask Ratio: {epoch_mask_ratio*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Val F1: {val_f1:.4f}")
        if epoch % 20 == 0:
            free_thresh.log_thresholds(epoch, CLASS_NAMES)
            
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_acc': val_acc,
                'val_f1': val_f1,
                'mask_ratio': epoch_mask_ratio
            }, HEMA_AUG_ONLY_CHECKPOINT)
            print(f"  ✔ New best model saved to {HEMA_AUG_ONLY_CHECKPOINT} (Val Acc: {val_acc*100:.2f}%)")

training_duration = time.time() - start_time
print(f"\nHema-Aug training completed in {training_duration/60:.2f} minutes!")
print(f"Best Validation Accuracy achieved: {best_val_acc*100:.2f}%")

In [ ]:
plot_threshold_evolution(free_thresh, CLASS_NAMES)

In [ ]:
# ============================================================
# SECTION 6.3 — Final Phase 6 Evaluation
# ============================================================

print("\n--- Running Final Evaluation on Test Set ---")

# Load best checkpoint
checkpoint = torch.load(HEMA_AUG_ONLY_CHECKPOINT)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best checkpoint from epoch {checkpoint['epoch']} with Val Acc {checkpoint['val_acc']*100:.2f}%")

# Evaluate on test set
test_acc, test_f1, true_labels, pred_labels = evaluate_model(model, test_loader, device)

print("\n=== Phase 6 (Hema-Aug Only) Test Set Performance ===")
print(f"Overall Test Accuracy : {test_acc*100:.2f}%")
print(f"Macro F1-Score        : {test_f1:.4f}")
print("======================================================")

print("\nDetailed Per-Class Performance:")
print(classification_report(true_labels, pred_labels, target_names=CLASS_NAMES))

# Define an automatic, descriptive filename for saving
save_filename = f"{run_name.lower().replace(' ', '_').replace('+', 'and')}_confusion_matrix.png"

# Generate the beautiful colorful plot
print("\nGenerating Confusion Matrix Plot...")
plot_colorful_confusion_matrix(
    true_labels=true_labels, 
    pred_labels=pred_labels, 
    class_names=CLASS_NAMES, 
    run_name=run_name,
    save_path=save_filename
)

log_run_results(
    run_name=run_name,
    true_labels=true_labels,
    pred_labels=pred_labels,
    test_acc=test_acc,
    test_f1=test_f1
)

## Distribution Alignment

In [ ]:
# ============================================================
# CONFIGURATION TOGGLE: Set this before running
# ============================================================
# FALSE = Phase 7a: Standard FixMatch + DA (uses RandAugment)
# TRUE  = Phase 7b: Full HemaMatch (uses Hema-Aug + DA)
USE_HEMA_AUG = False

In [ ]:
# ============================================================
# Dynamic Class Prior Estimation
# ============================================================

# Dynamically estimate the class prior p(y|X) from your labeled training set [D.1]
labeled_labels = []
for _, labels in labeled_loader:
    labeled_labels.extend(labels.squeeze().numpy())

unique_classes, counts = np.unique(labeled_labels, return_counts=True)
prior_counts = np.zeros(8)
for u, c in zip(unique_classes, counts):
    prior_counts[int(u)] = c

# Normalize prior frequencies and move to device [D.1]
class_prior = torch.tensor(prior_counts / prior_counts.sum(), dtype=torch.float32).to(device)
print(f"Estimated Class Prior p(y|X) for 1% split: {class_prior.cpu().numpy()}")

# Select the appropriate dataloader cycle and checkpoint name
if USE_HEMA_AUG:
    active_unlabeled_cycle = hema_unlabeled_cycle  # Hema-Aug loader
    CHECKPOINT_NAME = "hemamatch_full_best.pth"
    run_name = "Full HemaMatch (Hema-Aug + DA)"
else:
    active_unlabeled_cycle = unlabeled_cycle       # Standard RandAugment loader
    CHECKPOINT_NAME = "fixmatch_da_best.pth"
    run_name = "Standard FixMatch + DA"

In [ ]:
# ============================================================
# Model & Training Setup
# ============================================================

# Freshly re-initialize model for Phase 7 to ensure clean weights [3.1]
model = get_resnet18_backbone(num_classes=8).to(device)

criterion_supervised = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    model.parameters(), 
    lr=0.03, 
    momentum=0.9, 
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)

# Initialize the running mean prediction vector p_m(y|U) [D.1, 7.1]
running_mean = None


In [ ]:
# ============================================================
# Training Loop with Distribution Alignment
# ============================================================

print(f"\n--- Starting Training: {run_name} ---")
best_val_acc = 0.0
start_time = time.time()
# Fresh FreeMatch instance for this condition
free_thresh = FreeMatchThreshold(
num_classes=8, momentum=0.9, device=device)
running_mean = None  # reset DA running mean too

for epoch in range(1, 301):
    model.train()
    
    running_loss_s = 0.0
    running_loss_u = 0.0
    running_mask_ratio = 0.0
    steps_in_epoch = 0
    
    for (labeled_imgs, labels), (weak_unlabeled, strong_unlabeled) in zip(labeled_loader, active_unlabeled_cycle):
        
        labeled_imgs = labeled_imgs.to(device)
        labels = labels.squeeze().long().to(device)
        
        weak_unlabeled = weak_unlabeled.to(device)
        strong_unlabeled = strong_unlabeled.to(device)
        
        # Step 1: Supervised Loss
        logits_labeled = model(labeled_imgs)
        loss_supervised = criterion_supervised(logits_labeled, labels)
        
        # Step 2: Pseudo-Label Generation with Distribution Alignment [D.1, 7.1]
        with torch.no_grad():
            logits_weak = model(weak_unlabeled)
            probs_weak = torch.softmax(logits_weak, dim=1)
            
            # Update running mean p_m(y|U) using EMA momentum of 0.9 [D.1, 7.1]
            if running_mean is None:
                running_mean = probs_weak.mean(dim=0).detach()
            else:
                running_mean = 0.9 * running_mean + 0.1 * probs_weak.mean(dim=0).detach()
                
            # Distribution Alignment scaling: q_tilde = q * (prior / running_mean) [D.1, 7.1]
            probs_aligned = probs_weak * (class_prior / (running_mean + 1e-6))
            probs_aligned = probs_aligned / probs_aligned.sum(dim=1, keepdim=True)  # Re-normalize to sum to 1 [D.1]
            
            # Update FreeMatch thresholds using aligned probs
            free_thresh.update(probs_aligned)
            
            # FreeMatch adaptive mask on aligned probs
            confidence, pseudo_labels = probs_aligned.max(dim=1)
            mask = free_thresh.get_mask(probs_aligned, pseudo_labels)
            
        # Step 3: Consistency Loss on Strong Unlabeled
        logits_strong = model(strong_unlabeled)
        raw_loss_u = F.cross_entropy(logits_strong, pseudo_labels, reduction='none')
        loss_unsupervised = (raw_loss_u * mask).mean()
        
        # Step 4: Combined Loss with Lambda Ramp-Up
        lambda_u = get_lambda_ramp(epoch, ramp_epochs=20)
        total_loss = loss_supervised + lambda_u * loss_unsupervised
        
        # Gradient Update
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        
        running_loss_s += loss_supervised.item()
        running_loss_u += loss_unsupervised.item()
        running_mask_ratio += mask.mean().item()
        steps_in_epoch += 1
        
    scheduler.step()
    
    epoch_loss_s = running_loss_s / steps_in_epoch
    epoch_loss_u = running_loss_u / steps_in_epoch
    epoch_mask_ratio = running_mask_ratio / steps_in_epoch
    
    # Evaluate every 10 epochs [5.1]
    if epoch % 10 == 0 or epoch == 1:
        val_acc, val_f1, _, _ = evaluate_model(model, val_loader, device)
        print(f"Epoch {epoch:03d}/{EPOCHS} | Loss S: {epoch_loss_s:.4f} | Loss U: {epoch_loss_u:.4f} | "
              f"Mask Ratio: {epoch_mask_ratio*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Val F1: {val_f1:.4f}")
        if epoch % 20 == 0:
            free_thresh.log_thresholds(epoch, CLASS_NAMES)
            print(f"  [DA Running Mean]: {running_mean.cpu().numpy().round(4)}")
        
        
        # Log running_mean distribution convergence check [7.1]
        if epoch % 50 == 0:
            print(f"  [EMA Running Mean p_m(y|U) Convergence Check]:\n  {running_mean.cpu().numpy()}")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_acc': val_acc,
                'val_f1': val_f1,
                'mask_ratio': epoch_mask_ratio,
                'running_mean': running_mean
            }, CHECKPOINT_NAME)
            print(f"  ✔ New best model saved to {CHECKPOINT_NAME} (Val Acc: {val_acc*100:.2f}%)")

training_duration = time.time() - start_time
print(f"\nTraining completed in {training_duration/60:.2f} minutes!")
print(f"Best Validation Accuracy achieved: {best_val_acc*100:.2f}%")


In [ ]:
plot_threshold_evolution(free_thresh, CLASS_NAMES)

In [ ]:
# ============================================================
# Final Test Evaluation
# ============================================================

print(f"\n--- Running Final Evaluation on Test Set: {run_name} ---")

# Load best checkpoint
checkpoint = torch.load(CHECKPOINT_NAME)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best checkpoint from epoch {checkpoint['epoch']} with Val Acc {checkpoint['val_acc']*100:.2f}%")

# Evaluate on test set
test_acc, test_f1, true_labels, pred_labels = evaluate_model(model, test_loader, device)

print(f"\n=== {run_name} Test Set Performance ===")
print(f"Overall Test Accuracy : {test_acc*100:.2f}%")
print(f"Macro F1-Score        : {test_f1:.4f}")
print("======================================================")

print("\nDetailed Per-Class Performance:")
print(classification_report(true_labels, pred_labels, target_names=CLASS_NAMES))

# Define an automatic, descriptive filename for saving
save_filename = f"{run_name.lower().replace(' ', '_').replace('+', 'and')}_confusion_matrix.png"

# Generate the beautiful colorful plot
print("\nGenerating Confusion Matrix Plot...")
plot_colorful_confusion_matrix(
    true_labels=true_labels, 
    pred_labels=pred_labels, 
    class_names=CLASS_NAMES, 
    run_name=run_name,
    save_path=save_filename
)

log_run_results(
    run_name=run_name,
    true_labels=true_labels,
    pred_labels=pred_labels,
    test_acc=test_acc,
    test_f1=test_f1
)

## HemaMatch + DA

In [ ]:
USE_HEMA_AUG = True

In [ ]:
# ============================================================
# Dynamic Class Prior Estimation
# ============================================================

# Dynamically estimate the class prior p(y|X) from your labeled training set [D.1]
labeled_labels = []
for _, labels in labeled_loader:
    labeled_labels.extend(labels.squeeze().numpy())

unique_classes, counts = np.unique(labeled_labels, return_counts=True)
prior_counts = np.zeros(8)
for u, c in zip(unique_classes, counts):
    prior_counts[int(u)] = c

# Normalize prior frequencies and move to device [D.1]
class_prior = torch.tensor(prior_counts / prior_counts.sum(), dtype=torch.float32).to(device)
print(f"Estimated Class Prior p(y|X) for 1% split: {class_prior.cpu().numpy()}")

# Select the appropriate dataloader cycle and checkpoint name
if USE_HEMA_AUG:
    active_unlabeled_cycle = hema_unlabeled_cycle  # Hema-Aug loader
    CHECKPOINT_NAME = "hemamatch_full_best.pth"
    run_name = "Full HemaMatch (Hema-Aug + DA)"
else:
    active_unlabeled_cycle = unlabeled_cycle       # Standard RandAugment loader
    CHECKPOINT_NAME = "fixmatch_da_best.pth"
    run_name = "Standard FixMatch + DA"

In [ ]:
# ============================================================
# Model & Training Setup
# ============================================================

# Freshly re-initialize model for Phase 7 to ensure clean weights [3.1]
model = get_resnet18_backbone(num_classes=8).to(device)

criterion_supervised = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    model.parameters(), 
    lr=0.03, 
    momentum=0.9, 
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)

# Initialize the running mean prediction vector p_m(y|U) [D.1, 7.1]
running_mean = None

In [ ]:
# Training Loop with Distribution Alignment
# ============================================================

print(f"\n--- Starting Training: {run_name} ---")
best_val_acc = 0.0
start_time = time.time()
free_thresh.update(probs_aligned)
mask = free_thresh.get_mask(probs_aligned, pseudo_labels)

for epoch in range(1, 301):
    model.train()
    
    running_loss_s = 0.0
    running_loss_u = 0.0
    running_mask_ratio = 0.0
    steps_in_epoch = 0
    
    for (labeled_imgs, labels), (weak_unlabeled, strong_unlabeled) in zip(labeled_loader, active_unlabeled_cycle):
        
        labeled_imgs = labeled_imgs.to(device)
        labels = labels.squeeze().long().to(device)
        
        weak_unlabeled = weak_unlabeled.to(device)
        strong_unlabeled = strong_unlabeled.to(device)
        
        # Step 1: Supervised Loss
        logits_labeled = model(labeled_imgs)
        loss_supervised = criterion_supervised(logits_labeled, labels)
        
        # Step 2: Pseudo-Label Generation with Distribution Alignment [D.1, 7.1]
        with torch.no_grad():
            logits_weak = model(weak_unlabeled)
            probs_weak = torch.softmax(logits_weak, dim=1)
            
            # Update running mean p_m(y|U) using EMA momentum of 0.9 [D.1, 7.1]
            if running_mean is None:
                running_mean = probs_weak.mean(dim=0).detach()
            else:
                running_mean = 0.9 * running_mean + 0.1 * probs_weak.mean(dim=0).detach()
                
            # Distribution Alignment scaling: q_tilde = q * (prior / running_mean) [D.1, 7.1]
            probs_aligned = probs_weak * (class_prior / (running_mean + 1e-6))
            probs_aligned = probs_aligned / probs_aligned.sum(dim=1, keepdim=True)  # Re-normalize to sum to 1 [D.1]
            
            # Update FreeMatch thresholds using aligned probs
            free_thresh.update(probs_aligned)
            
            # FreeMatch adaptive mask on aligned probs
            confidence, pseudo_labels = probs_aligned.max(dim=1)
            mask = free_thresh.get_mask(probs_aligned, pseudo_labels)
                        
        # Step 3: Consistency Loss on Strong Unlabeled
        logits_strong = model(strong_unlabeled)
        raw_loss_u = F.cross_entropy(logits_strong, pseudo_labels, reduction='none')
        loss_unsupervised = (raw_loss_u * mask).mean()
        
        # Step 4: Combined Loss with Lambda Ramp-Up
        lambda_u = get_lambda_ramp(epoch, ramp_epochs=20)
        total_loss = loss_supervised + lambda_u * loss_unsupervised
        
        # Gradient Update
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        
        running_loss_s += loss_supervised.item()
        running_loss_u += loss_unsupervised.item()
        running_mask_ratio += mask.mean().item()
        steps_in_epoch += 1
        
    scheduler.step()
    
    epoch_loss_s = running_loss_s / steps_in_epoch
    epoch_loss_u = running_loss_u / steps_in_epoch
    epoch_mask_ratio = running_mask_ratio / steps_in_epoch
    
    # Evaluate every 10 epochs [5.1]
    if epoch % 10 == 0 or epoch == 1:
        val_acc, val_f1, _, _ = evaluate_model(model, val_loader, device)
        print(f"Epoch {epoch:03d}/{300} | Loss S: {epoch_loss_s:.4f} | Loss U: {epoch_loss_u:.4f} | "
              f"Mask Ratio: {epoch_mask_ratio*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Val F1: {val_f1:.4f}")
        
        if epoch % 20 == 0:
            free_thresh.log_thresholds(epoch, CLASS_NAMES)
        
        # Log running_mean distribution convergence check [7.1]
        if epoch % 50 == 0:
            print(f"  [EMA Running Mean p_m(y|U) Convergence Check]:\n  {running_mean.cpu().numpy()}")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_acc': val_acc,
                'val_f1': val_f1,
                'mask_ratio': epoch_mask_ratio,
                'running_mean': running_mean
            }, CHECKPOINT_NAME)
            print(f"  ✔ New best model saved to {CHECKPOINT_NAME} (Val Acc: {val_acc*100:.2f}%)")

training_duration = time.time() - start_time
print(f"\nTraining completed in {training_duration/60:.2f} minutes!")
print(f"Best Validation Accuracy achieved: {best_val_acc*100:.2f}%")


In [ ]:
plot_threshold_evolution(free_thresh, CLASS_NAMES)

In [ ]:
# ============================================================
# Final Test Evaluation
# ============================================================

print(f"\n--- Running Final Evaluation on Test Set: {run_name} ---")

# Load best checkpoint
checkpoint = torch.load(CHECKPOINT_NAME)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best checkpoint from epoch {checkpoint['epoch']} with Val Acc {checkpoint['val_acc']*100:.2f}%")

# Evaluate on test set
test_acc, test_f1, true_labels, pred_labels = evaluate_model(model, test_loader, device)

print(f"\n=== {run_name} Test Set Performance ===")
print(f"Overall Test Accuracy : {test_acc*100:.2f}%")
print(f"Macro F1-Score        : {test_f1:.4f}")
print("======================================================")

print("\nDetailed Per-Class Performance:")
print(classification_report(true_labels, pred_labels, target_names=CLASS_NAMES))

# Define an automatic, descriptive filename for saving
save_filename = f"{run_name.lower().replace(' ', '_').replace('+', 'and')}_confusion_matrix.png"

# Generate the beautiful colorful plot
print("\nGenerating Confusion Matrix Plot...")
plot_colorful_confusion_matrix(
    true_labels=true_labels, 
    pred_labels=pred_labels, 
    class_names=CLASS_NAMES, 
    run_name=run_name,
    save_path=save_filename
)

log_run_results(
    run_name=run_name,
    true_labels=true_labels,
    pred_labels=pred_labels,
    test_acc=test_acc,
    test_f1=test_f1
)